# 00 - Setup SQL Server

Cria o banco de dados `LojaVirtualDB` no SQL Server e carrega as tabelas a partir dos CSVs da pasta `data/`.

## Pré-requisitos
- Docker rodando com o container `sqlserver-2022` ativo (`docker compose up -d`)
- ODBC Driver 18 for SQL Server instalado
- Ambiente virtual ativado com as dependências instaladas

## Schema

```sql
CREATE TABLE clientes (id INT, nome NVARCHAR(100), email NVARCHAR(100), cidade NVARCHAR(100))
CREATE TABLE produtos  (id INT, nome NVARCHAR(100), categoria NVARCHAR(100), preco DECIMAL(10,2), estoque INT)
CREATE TABLE pedidos   (id INT, cliente_id INT, produto_id INT, quantidade INT, valor_total DECIMAL(10,2), data_pedido DATE, status NVARCHAR(50))
```


In [ ]:
import pyodbc
import pandas as pd
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

DRIVER   = "{ODBC Driver 18 for SQL Server}"
SERVER   = f"{os.getenv('SQLSERVER_HOST')},{os.getenv('SQLSERVER_PORT')}"
USER     = os.getenv("SQLSERVER_USER")
PASSWORD = os.getenv("SQLSERVER_PASSWORD")
DATABASE = os.getenv("SQLSERVER_DB")

# Localiza a pasta data/ independente de onde o Jupyter foi iniciado
cwd = os.getcwd()
DATA_DIR = os.path.join(os.path.dirname(cwd), "data") if os.path.basename(cwd) == "notebooks" else os.path.join(cwd, "data")
print(f"Pasta de dados: {DATA_DIR}")

In [ ]:
# Cria o banco de dados caso não exista
conn_master = pyodbc.connect(
    f"DRIVER={DRIVER};SERVER={SERVER};DATABASE=master;UID={USER};PWD={PASSWORD};TrustServerCertificate=yes;",
    autocommit=True,
)
conn_master.cursor().execute(
    f"IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{DATABASE}') CREATE DATABASE {DATABASE}"
)
conn_master.close()
print(f"Banco '{DATABASE}' pronto.")

In [ ]:
# Conecta ao banco e recria as tabelas
conn = pyodbc.connect(
    f"DRIVER={DRIVER};SERVER={SERVER};DATABASE={DATABASE};UID={USER};PWD={PASSWORD};TrustServerCertificate=yes;",
    autocommit=True,
)
cur = conn.cursor()

cur.execute("IF OBJECT_ID('pedidos',  'U') IS NOT NULL DROP TABLE pedidos")
cur.execute("IF OBJECT_ID('produtos', 'U') IS NOT NULL DROP TABLE produtos")
cur.execute("IF OBJECT_ID('clientes', 'U') IS NOT NULL DROP TABLE clientes")

cur.execute("""
    CREATE TABLE clientes (
        id      INT PRIMARY KEY,
        nome    NVARCHAR(100),
        email   NVARCHAR(100),
        cidade  NVARCHAR(100)
    )
""")

cur.execute("""
    CREATE TABLE produtos (
        id        INT PRIMARY KEY,
        nome      NVARCHAR(100),
        categoria NVARCHAR(100),
        preco     DECIMAL(10,2),
        estoque   INT
    )
""")

cur.execute("""
    CREATE TABLE pedidos (
        id          INT PRIMARY KEY,
        cliente_id  INT,
        produto_id  INT,
        quantidade  INT,
        valor_total DECIMAL(10,2),
        data_pedido DATE,
        status      NVARCHAR(50),
        FOREIGN KEY (cliente_id) REFERENCES clientes(id),
        FOREIGN KEY (produto_id) REFERENCES produtos(id)
    )
""")

print("Tabelas criadas.")

In [ ]:
# Lê os CSVs e insere os dados nas tabelas
def inserir(cursor, tabela, df):
    cols = ", ".join(df.columns)
    params = ", ".join(["?"] * len(df.columns))
    sql = f"INSERT INTO {tabela} ({cols}) VALUES ({params})"
    for row in df.itertuples(index=False, name=None):
        cursor.execute(sql, row)

clientes = pd.read_csv(os.path.join(DATA_DIR, "clientes.csv"))
produtos  = pd.read_csv(os.path.join(DATA_DIR, "produtos.csv"))
pedidos   = pd.read_csv(os.path.join(DATA_DIR, "pedidos.csv"))

inserir(cur, "clientes", clientes)
inserir(cur, "produtos",  produtos)
inserir(cur, "pedidos",   pedidos)

conn.close()
print(f"Inseridos: {len(clientes)} clientes | {len(produtos)} produtos | {len(pedidos)} pedidos")

In [ ]:
# Verifica os dados inseridos
conn = pyodbc.connect(
    f"DRIVER={DRIVER};SERVER={SERVER};DATABASE={DATABASE};UID={USER};PWD={PASSWORD};TrustServerCertificate=yes;"
)

for tabela in ["clientes", "produtos", "pedidos"]:
    df = pd.read_sql(f"SELECT * FROM {tabela}", conn)
    print(f"\n=== {tabela.upper()} ({len(df)} registros) ===")
    print(df.to_string(index=False))

conn.close()